In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

StatementMeta(, fd4e0b1a-024f-49a3-8747-4db9827f3078, 3, Finished, Available, Finished, False)

In [ ]:
RAW_TABLE = "dbo.raw_market_ticks"
SILVER_TABLE = "dbo.silver_market_ticks"

StatementMeta(, fd4e0b1a-024f-49a3-8747-4db9827f3078, 5, Finished, Available, Finished, False)

In [ ]:
RAW_TABLE = "bronze_trades"
SILVER_TABLE = "silver_market_ticks"

raw_df = spark.read.table(RAW_TABLE)

display(raw_df.limit(10))

StatementMeta(, fd4e0b1a-024f-49a3-8747-4db9827f3078, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e4b5b516-9801-4a47-a598-028293eec9e9)

In [ ]:
silver_df = (
    raw_df
    .select(
        F.to_timestamp("event_time").alias("event_time"),
        F.col("symbol").cast("string").alias("symbol"),
        F.col("price").cast("double").alias("price"),
        F.col("last_size").cast("double").alias("last_size"),
        F.col("best_bid").cast("double").alias("best_bid"),
        F.col("best_ask").cast("double").alias("best_ask"),
        F.col("side").cast("string").alias("side"),
        F.col("exchange").cast("string").alias("exchange"),
        F.to_timestamp("ingest_utc").alias("ingest_utc")
    )
    .where(F.col("event_time").isNotNull())
    .where(F.col("symbol").isNotNull())
    .where(F.col("price").isNotNull())
    .dropDuplicates(["exchange", "symbol", "event_time", "price", "last_size"])
)

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SILVER_TABLE)

display(silver_df.limit(10))

StatementMeta(, fd4e0b1a-024f-49a3-8747-4db9827f3078, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0230e8a1-8e7a-4489-9db9-bc4f5dc4b534)

In [ ]:
OHLC_TABLE = "ohlc_1min"

ticks_df = spark.read.table(SILVER_TABLE)

ticks_df = (
    ticks_df
    .where(F.col("event_time").isNotNull())
    .where(F.col("price").isNotNull())
    .withColumn("minute_ts", F.date_trunc("minute", F.col("event_time")))
)

StatementMeta(, fd4e0b1a-024f-49a3-8747-4db9827f3078, 9, Finished, Available, Finished, False)

In [ ]:
w_open = Window.partitionBy("symbol", "minute_ts").orderBy(F.col("event_time").asc())

open_df = (
    ticks_df
    .withColumn("rn_open", F.row_number().over(w_open))
    .where(F.col("rn_open") == 1)
    .select(
        "symbol",
        "minute_ts",
        F.col("price").alias("Open")
    )
)

StatementMeta(, fd4e0b1a-024f-49a3-8747-4db9827f3078, 10, Finished, Available, Finished, False)

In [ ]:
w_close = Window.partitionBy("symbol", "minute_ts").orderBy(F.col("event_time").desc())

close_df = (
    ticks_df
    .withColumn("rn_close", F.row_number().over(w_close))
    .where(F.col("rn_close") == 1)
    .select(
        "symbol",
        "minute_ts",
        F.col("price").alias("Close")
    )
)

StatementMeta(, fd4e0b1a-024f-49a3-8747-4db9827f3078, 11, Finished, Available, Finished, False)

In [ ]:
hlv_df = (
    ticks_df
    .groupBy("symbol", "minute_ts")
    .agg(
        F.max("price").alias("High"),
        F.min("price").alias("Low"),
        F.sum(F.coalesce(F.col("last_size"), F.lit(0.0))).alias("Volume"),
        F.count("*").alias("Trades")
    )
)

StatementMeta(, fd4e0b1a-024f-49a3-8747-4db9827f3078, 12, Finished, Available, Finished, False)

In [ ]:
ohlc_df = (
    hlv_df
    .join(open_df, ["symbol", "minute_ts"], "inner")
    .join(close_df, ["symbol", "minute_ts"], "inner")
    .select(
        "symbol",
        "minute_ts",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume",
        "Trades"
    )
    .orderBy("symbol", "minute_ts")
)

ohlc_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(OHLC_TABLE)

display(ohlc_df.limit(20))

StatementMeta(, fd4e0b1a-024f-49a3-8747-4db9827f3078, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, aa5085dd-5e1a-49c5-9656-4a280997cce3)

In [ ]:
FEATURE_TABLE = "features_market_1min"

ohlc_df = spark.read.table(OHLC_TABLE)

w = Window.partitionBy("symbol").orderBy("minute_ts")

features_df = (
    ohlc_df
    .withColumn("prev_close", F.lag("Close", 1).over(w))
    .withColumn(
        "return_1m",
        (F.col("Close") - F.col("prev_close")) / F.col("prev_close")
    )
    .withColumn(
        "sma_5",
        F.avg("Close").over(w.rowsBetween(-4, 0))
    )
    .withColumn(
        "sma_20",
        F.avg("Close").over(w.rowsBetween(-19, 0))
    )
    .withColumn(
        "volatility_20",
        F.stddev("return_1m").over(w.rowsBetween(-19, 0))
    )
    .withColumn(
        "volume_sma_20",
        F.avg("Volume").over(w.rowsBetween(-19, 0))
    )
    .withColumn(
        "volume_ratio",
        F.col("Volume") / F.col("volume_sma_20")
    )
)

features_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(FEATURE_TABLE)

display(features_df.limit(20))

StatementMeta(, fd4e0b1a-024f-49a3-8747-4db9827f3078, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cc3d495f-7989-48f6-816d-4ac0654c324a)

In [ ]:
backtest_df = (
    features_df
    .where(F.col("sma_5").isNotNull())
    .where(F.col("sma_20").isNotNull())
    .withColumn(
        "signal",
        F.when(F.col("sma_5") > F.col("sma_20"), F.lit(1)).otherwise(F.lit(0))
    )
)

w = Window.partitionBy("symbol").orderBy("minute_ts")

backtest_df = (
    backtest_df
    .withColumn("prev_signal", F.lag("signal", 1).over(w))
    .withColumn(
        "strategy_return",
        F.col("prev_signal") * F.col("return_1m")
    )
)

result_df = (
    backtest_df
    .groupBy("symbol")
    .agg(
        F.sum("strategy_return").alias("total_strategy_return"),
        F.avg("strategy_return").alias("avg_strategy_return"),
        F.stddev("strategy_return").alias("risk"),
        F.count("*").alias("periods")
    )
    .withColumn(
        "simple_sharpe_like_ratio",
        F.col("avg_strategy_return") / F.col("risk")
    )
)

display(result_df)

StatementMeta(, fd4e0b1a-024f-49a3-8747-4db9827f3078, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 358004cc-0f8f-4624-aa24-1e0f2523d1b4)